<a href="https://www.kaggle.com/code/rabiabatool021/final-project?scriptVersionId=344433386" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Brain Tumor Detection (MRI Scans)
### CNN (VGG16 / ResNet50 Transfer Learning) + Classical Supervised ML

This notebook builds and compares **4 models** for classifying brain MRI scans into 4 classes: **glioma, meningioma, pituitary, notumor**.

1. **Baseline ANN** (no convolution) — deep learning module baseline
2. **CNN built from scratch** — deep learning module
3. **Transfer Learning (VGG16 / ResNet50)** — deep learning module, main model
4. **Classical Supervised ML (SVM, Random Forest, Logistic Regression)** trained on CNN-extracted features — explicit supervised ML module component

Dataset path: `/kaggle/input/datasets/bilalakgz/brain-tumor-mri-dataset`



## 1. Imports & Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import (Input, Dense, Flatten, Dropout, Conv2D,
                                      MaxPooling2D, BatchNormalization,
                                      GlobalAveragePooling2D)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2. Explore Dataset Structure


brain_tumor_dataset/brain_tumor_classification/
├── Training/
│   ├── no_tumor/
│   ├── pituitary_tumor/
│   ├── meningioma_tumor/
│   └── glioma_tumor/
└── Testing/
    ├── no_tumor/
    ├── pituitary_tumor/
    ├── meningioma_tumor/
    └── glioma_tumor/

In [ ]:
DATASET_ROOT = '/kaggle/input/datasets/bilalakgz/brain-tumor-mri-dataset'

for root, dirs, files in os.walk(DATASET_ROOT):
    depth = root.replace(DATASET_ROOT, '').count(os.sep)
    if depth <= 2:
        print(('  ' * depth) + os.path.basename(root) + '/', '->',
              f'{len(dirs)} subfolders, {len(files)} files')

In [ ]:
CLASSIFICATION_ROOT = '/kaggle/input/datasets/bilalakgz/brain-tumor-mri-dataset/brain_tumor_dataset/brain_tumor_classification'

for root, dirs, files in os.walk(CLASSIFICATION_ROOT):
    depth = root.replace(CLASSIFICATION_ROOT, '').count(os.sep)
    print(('  ' * depth) + os.path.basename(root) + '/', '->',
          f'{len(dirs)} subfolders, {len(files)} files')

## 3. Configuration
**Update `TRAIN_DIR` / `TEST_DIR` below based on what Step 2 printed.** If the dataset has a single folder with all classes (no separate Training/Testing split), see the alternate cell further down to split it yourself.

In [ ]:
# --- Adjust these two paths based on the folder structure printed above ---
TRAIN_DIR = f'{DATASET_ROOT}/brain_tumor_dataset/brain_tumor_classification/Training'
TEST_DIR  = f'{DATASET_ROOT}/brain_tumor_dataset/brain_tumor_classification/Testing'

IMAGE_SIZE = (224, 224)   # required input size for VGG16 / ResNet50
BATCH_SIZE = 32
NUM_CLASSES = 4
CLASS_NAMES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
BACKBONE = 'resnet50'      # or 'vgg16'
EPOCHS = 20

WORK_DIR = '/kaggle/working'
os.makedirs(WORK_DIR, exist_ok=True)           

In [ ]:
# OPTIONAL: only run this if there is no Training/Testing split in the dataset.
# import shutil, random
# from sklearn.model_selection import train_test_split
#
# SPLIT_SRC = DATASET_ROOT   # folder containing class subfolders directly
# TRAIN_DIR = f'{WORK_DIR}/Training'
# TEST_DIR  = f'{WORK_DIR}/Testing'
#
# for cls in CLASS_NAMES:
#     files = os.listdir(f'{SPLIT_SRC}/{cls}')
#     train_files, test_files = train_test_split(files, test_size=0.2, random_state=42)
#     os.makedirs(f'{TRAIN_DIR}/{cls}', exist_ok=True)
#     os.makedirs(f'{TEST_DIR}/{cls}', exist_ok=True)
#     for f in train_files:
#         shutil.copy(f'{SPLIT_SRC}/{cls}/{f}', f'{TRAIN_DIR}/{cls}/{f}')
#     for f in test_files:
#         shutil.copy(f'{SPLIT_SRC}/{cls}/{f}', f'{TEST_DIR}/{cls}/{f}')
# print('Custom split created at', WORK_DIR)

## 4. Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, cls in enumerate(CLASS_NAMES):
    cls_dir = f'{TRAIN_DIR}/{cls}'
    sample_file = os.listdir(cls_dir)[0]
    img = plt.imread(f'{cls_dir}/{sample_file}')
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(cls)
    axes[i].axis('off')
plt.suptitle('Sample MRI Scans per Class')
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/sample_images.png')
plt.show()

## 5. Data Generators
Training data is split 85/15 into train/validation. Light augmentation is applied only to the training set. A separate, non-shuffled generator is created for feature extraction later (needed to keep image order aligned with labels).

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.15,
)
test_datagen = ImageDataGenerator(rescale=1.0/255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES,
    subset='training', shuffle=True, seed=42)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES,
    subset='validation', shuffle=False, seed=42)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES, shuffle=False)

# Non-shuffled, non-augmented generators (for classical ML feature extraction)
feat_train_gen = test_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES, shuffle=False)

print('Class indices:', train_gen.class_indices)

## 6. Model 1 — Baseline ANN (no convolution)
A plain fully-connected network with no convolutional layers. Used purely as a comparison baseline to show why CNNs are needed for image data.

In [ ]:
def build_ann_model(input_shape=(224,224,3), num_classes=NUM_CLASSES):
    inputs = Input(shape=input_shape, name='image_input')
    x = Flatten()(inputs)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax', name='prediction')(x)
    return Model(inputs=inputs, outputs=outputs, name='baseline_ann')

early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

ann_model = build_ann_model()
ann_model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
ann_model.summary()

In [ ]:
history_ann = ann_model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS,
                             callbacks=[early_stop])
ann_model.save(f'{WORK_DIR}/ann_model.keras')

## 7. Model 2 — CNN Built From Scratch
A custom convolutional network (4 conv blocks) trained from random initialization — no pretrained weights. This is the middle step in the ANN -> CNN -> Transfer Learning progression.

In [ ]:
def build_cnn_model(input_shape=(224,224,3), num_classes=NUM_CLASSES):
    inputs = Input(shape=input_shape, name='image_input')
    x = Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(num_classes, activation='softmax', name='prediction')(x)
    return Model(inputs=inputs, outputs=outputs, name='cnn_from_scratch')

cnn_model = build_cnn_model()
cnn_model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()

In [ ]:
history_cnn = cnn_model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS,
                             callbacks=[early_stop])
cnn_model.save(f'{WORK_DIR}/cnn_model.keras')

## 8. Model 3 — Transfer Learning (VGG16 / ResNet50)
Uses ImageNet-pretrained weights, with the last few backbone layers unfrozen for fine-tuning. This is the main deep learning model of the project.

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

# Separate generators just for the transfer learning model
transfer_train_datagen = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,   # <-- fix: proper ResNet50 preprocessing
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.15,
)
transfer_test_datagen = ImageDataGenerator(preprocessing_function=resnet_preprocess)

transfer_train_gen = transfer_train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES,
    subset='training', shuffle=True, seed=42)

transfer_val_gen = transfer_train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES,
    subset='validation', shuffle=False, seed=42)

transfer_test_gen = transfer_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES, shuffle=False)

In [ ]:
def build_transfer_model(backbone=BACKBONE, input_shape=(224,224,3),
                          num_classes=NUM_CLASSES, fine_tune_last_n=15):
    inputs = Input(shape=input_shape, name='image_input')
    if backbone == 'vgg16':
        base_model = VGG16(weights='imagenet', include_top=False, input_tensor=inputs)
    elif backbone == 'resnet50':
        base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=inputs)
    else:
        raise ValueError("backbone must be 'vgg16' or 'resnet50'")

    base_model.trainable = True
    for layer in base_model.layers[:-fine_tune_last_n]:
        layer.trainable = False

    x = GlobalAveragePooling2D(name='feature_vector')(base_model.output)
    y = Dense(256, activation='relu')(x)
    y = Dropout(0.4)(y)
    outputs = Dense(num_classes, activation='softmax', name='prediction')(y)
    return Model(inputs=inputs, outputs=outputs, name=f'transfer_{backbone}')

transfer_model = build_transfer_model()
transfer_model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
transfer_model.summary()



In [ ]:
history_transfer = transfer_model.fit(transfer_train_gen, validation_data=transfer_val_gen,
                                       epochs=EPOCHS, callbacks=[early_stop])
transfer_model.save(f'{WORK_DIR}/transfer_model.keras')
# non-shuffled version for feature extraction (uses same ResNet50 preprocessing)
transfer_feat_train_gen = transfer_test_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASS_NAMES, shuffle=False)

## 9. Model 4 — Classical Supervised ML on CNN-Extracted Features
This is the explicit **Supervised ML module** component: instead of letting the neural network classify directly, we use the fine-tuned transfer model as a **feature extractor** (its `feature_vector` layer output), then train classical supervised ML algorithms — **SVM, Random Forest, Logistic Regression** — on those features. This lets you directly compare deep learning classification vs classical ML classification on the same learned features.

In [ ]:
# Build a feature-extractor model that outputs the GlobalAveragePooling layer
feature_extractor = Model(inputs=transfer_model.input,
                           outputs=transfer_model.get_layer('feature_vector').output)

def extract_features(generator):
    generator.reset()
    features = feature_extractor.predict(generator, verbose=1)
    labels = generator.classes[:len(features)]
    return features, labels

print('Extracting training features...')
X_train_feat, y_train_feat = extract_features(transfer_feat_train_gen)

print('Extracting test features...')
X_test_feat, y_test_feat = extract_features(transfer_test_gen)

print('Feature shapes:', X_train_feat.shape, X_test_feat.shape)



In [ ]:
classical_models = {
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
}

classical_predictions = {}
for name, clf in classical_models.items():
    print(f'Training {name}...')
    clf.fit(X_train_feat, y_train_feat)
    preds = clf.predict(X_test_feat)
    classical_predictions[name] = preds
    print(f'{name} done.')

## 10. Evaluation & Model Comparison (Ablation Study)
Compares all models — ANN, CNN-from-scratch, Transfer Learning, and the three classical ML classifiers — on accuracy, precision, recall, and F1-score, plus confusion matrices for each.

In [ ]:
def get_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='macro'),
        'recall': recall_score(y_true, y_pred, average='macro'),
        'f1': f1_score(y_true, y_pred, average='macro'),
    }

def plot_cm(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/{filename}')
    plt.show()

results = {}

In [ ]:
# --- Deep learning models (predict via generators) ---
test_gen.reset()
y_true_dl = test_gen.classes

for name, model in [('ANN (baseline)', ann_model),
                     ('CNN (from scratch)', cnn_model),
                     ('Transfer Learning', transfer_model)]:
    test_gen.reset()
    probs = model.predict(test_gen)
    preds = np.argmax(probs, axis=1)
    results[name] = get_metrics(y_true_dl, preds)
    plot_cm(y_true_dl, preds, f'{name} - Confusion Matrix',
            f"cm_{name.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png")

# Transfer Learning — naya transfer_test_gen (ResNet50 preprocessing) ke sath
transfer_test_gen.reset()
y_true_transfer = transfer_test_gen.classes
probs = transfer_model.predict(transfer_test_gen)
preds = np.argmax(probs, axis=1)
results['Transfer Learning'] = get_metrics(y_true_transfer, preds)
plot_cm(y_true_transfer, preds, 'Transfer Learning - Confusion Matrix',
        'cm_transfer_learning.png')

In [ ]:
# --- Classical ML models (predict on extracted features) ---
for name, preds in classical_predictions.items():
    results[name] = get_metrics(y_test_feat, preds)
    plot_cm(y_test_feat, preds, f'{name} - Confusion Matrix',
            f"cm_{name.lower().replace(' ', '_')}.png")

In [ ]:
comparison_df = pd.DataFrame(results).T.round(4)
comparison_df = comparison_df.sort_values('accuracy', ascending=False)
print('=== Full Model Comparison ===')
print(comparison_df)
comparison_df.to_csv(f'{WORK_DIR}/model_comparison_results.csv')

comparison_df.plot(kind='bar', figsize=(11,5))
plt.title('Model Comparison: ANN vs CNN vs Transfer Learning vs Classical ML')
plt.ylabel('Score')
plt.ylim(0,1)
plt.xticks(rotation=20, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/model_comparison_chart.png')
plt.show()

## 11. Training Curves (Accuracy / Loss)
Useful for your report to show convergence behavior and check for overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for name, hist in [('ANN', history_ann), ('CNN', history_cnn), ('Transfer', history_transfer)]:
    axes[0].plot(hist.history['accuracy'], label=f'{name} train')
    axes[0].plot(hist.history['val_accuracy'], '--', label=f'{name} val')
    axes[1].plot(hist.history['loss'], label=f'{name} train')
    axes[1].plot(hist.history['val_loss'], '--', label=f'{name} val')

axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend(fontsize=8)
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/training_curves.png')
plt.show()

## 12. Conclusion & Report Notes

**Saved outputs (in `/kaggle/working/`, downloadable from the Output tab):**
- `ann_model.keras`, `cnn_model.keras`, `transfer_model.keras`
- `model_comparison_results.csv` — full metrics table
- `model_comparison_chart.png` — bar chart comparing all 6 models
- `cm_*.png` — confusion matrix for every model
- `training_curves.png` — accuracy/loss curves
- `sample_images.png` — sample MRI per class

**Suggested report structure:**
1. Introduction & Problem Statement
2. Dataset Description (source, classes, size, split)
3. Methodology
   - Preprocessing & augmentation
   - Deep learning progression: ANN -> CNN (scratch) -> Transfer Learning (VGG16/ResNet50)
   - Supervised ML: classical classifiers (SVM, Random Forest, Logistic Regression) on CNN-extracted features
4. Results — comparison table, bar chart, confusion matrices, training curves
5. Discussion — why transfer learning outperforms ANN/CNN-from-scratch; how classical ML on deep features compares
6. **Ethics & Limitations** — this is a proof-of-concept, not a clinical diagnostic tool; would require clinical validation, larger and more diverse datasets, and radiologist oversight before real-world use
7. Conclusion & Future Work

In [ ]:
# Teeno models ki weights .h5 format mein save karna
ann_model.save_weights('/kaggle/working/ann_model_weights.weights.h5')
cnn_model.save_weights('/kaggle/working/cnn_model_weights.weights.h5')
transfer_model.save_weights('/kaggle/working/transfer_model_weights.weights.h5')

print("Teeno models ki weights save ho gayi!")

In [ ]:
from tensorflow.keras.models import load_model

# Saved .keras models load karna
ann_model = load_model('/kaggle/working/ann_model.keras')
cnn_model = load_model('/kaggle/working/cnn_model.keras')
transfer_model = load_model('/kaggle/working/transfer_model.keras')

# Ab weights save karna
ann_model.save_weights('/kaggle/working/ann_model_weights.weights.h5')
cnn_model.save_weights('/kaggle/working/cnn_model_weights.weights.h5')
transfer_model.save_weights('/kaggle/working/transfer_model_weights.weights.h5')

print("Teeno models load ho kar weights save ho gayi!")